# Encoder–HistGB with visit_history encoder inputs

The baseline is the unchanged pipeline from features_selection.ipynb. The encoder receives
original imputed features, their missingness indicators and only the seven visit_history features:
elapsed time, past/future visit counts, ON/OFF/LEDD patient slopes and the median OFF–ON gap.
Other engineered groups remain available to HistGB only.

Architecture and training stay unchanged: two bidirectional Transformer encoder blocks,
four heads, width 64, no CLS token, and an internal 20% patient holdout for epoch selection.
Encoder preprocessing keeps only visit_history before final median imputation, fitted separately
inside the internal holdout and each full training fold. Frozen embeddings are appended to
HistGB's existing features.

Validation uses five disjoint patient folds and seeds 42, 43 and 44 (15 runs). This notebook only evaluates
validation performance; there is no final-training or submission-generation section.

## 1. Reuse the exact baseline and set training options

Only the five setup cells from features_selection.ipynb are executed, not its validation or
feature-selection experiments. Run this notebook from the project root, exploration, or exploration/stacking folder.
PyTorch is required: uv pip install --python .venv/bin/python torch.

In [1]:
import json
import os
from pathlib import Path
from copy import deepcopy
import torch
from torch import nn
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold

PROJECT_DIR = next(p for p in (Path.cwd(), *Path.cwd().parents)
                   if (p / "exploration/features_selection.ipynb").is_file())
reference = json.loads((PROJECT_DIR / "exploration/features_selection.ipynb").read_text())
previous_dir = Path.cwd()
try:
    os.chdir(PROJECT_DIR)
    for i in [1, 3, 4, 6, 8]:  # Imports/data, history, imputation, features, pipeline.
        exec("".join(reference["cells"][i]["source"]))
finally:
    os.chdir(previous_dir)
DATA_DIR = PROJECT_DIR / "data"
baseline = clone(pipeline)
VISIT_HISTORY = [
    "time_since_first_visit", "n_past_visits", "n_future_visits",
    "on_slope_patient", "off_slope_patient", "ledd_slope_patient", "off_on_median_patient",
]
encoder_prep = clone(baseline[:-1]).set_params(
    features_eng=FunctionTransformer(
        lambda df: df.join(build_visit_features(df)[VISIT_HISTORY]).drop(columns="patient_id")
    )
)

ENCODER = dict(width=64, heads=8, layers=3, dropout=0.13)
MAX_EPOCHS, PATIENCE, BATCH_SIZE = 100, 10, 128
LEARNING_RATE, WEIGHT_DECAY = 1e-3, 1e-4
SEEDS = [42, 43, 44]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else
                      "mps" if torch.backends.mps.is_available() else "cpu")
torch.set_num_threads(3)
print(f"Device: {DEVICE}; baseline: {baseline[-1]}")

Device: mps; baseline: HistGradientBoostingRegressor(l2_regularization=1.8115270802885095,
                              learning_rate=0.042592090635273853,
                              max_features=0.6078232265551354, max_iter=800,
                              max_leaf_nodes=62, min_samples_leaf=25,
                              random_state=42)


## 2. One sequence per patient

The encoder keeps original imputed features and adds only visit_history, followed by median
imputation and missingness indicators. Patient IDs are removed from its inputs; raw IDs and
ages only group and order visits. Padding and original row positions work as before.

In [2]:
def patient_sequences(raw, features, targets=None):
    visits = raw.reset_index(drop=True).sort_values(["patient_id", "age"])
    rows = [torch.tensor(group.index.to_numpy())
            for _, group in visits.groupby("patient_id", sort=False)]
    values = torch.tensor(np.asarray(features), dtype=torch.float32)
    labels = torch.tensor(np.zeros(len(raw)) if targets is None else np.asarray(targets),
                          dtype=torch.float32)
    return TensorDataset(
        pad_sequence([values[i] for i in rows], batch_first=True),
        pad_sequence([labels[i] for i in rows], batch_first=True),
        pad_sequence(rows, batch_first=True, padding_value=-1),
    )

## 3. Small bidirectional encoder

Dense input projection → two sequential TransformerEncoder layers → one embedding per visit.
Each layer already includes parallel attention heads, feed-forward layers, residual connections
and LayerNorm. The regression head is used for supervised training only.

In [3]:
class VisitEncoder(nn.Module):
    def __init__(self, n_features, width=64, heads=4, layers=2, dropout=0.1):
        super().__init__()
        self.projection = nn.Linear(n_features, width)
        layer = nn.TransformerEncoderLayer(
            d_model=width, nhead=heads, dim_feedforward=3 * width,
            dropout=dropout, activation="gelu", batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=layers,
                                             enable_nested_tensor=False)
        # TransformerEncoder copies its layer: give each block independent weights.
        for parameter in self.encoder.parameters():
            if parameter.ndim > 1:
                nn.init.xavier_uniform_(parameter)
        self.head = nn.Linear(width, 1)

    def encode(self, values, padding):
        return self.encoder(self.projection(values), src_key_padding_mask=padding,
                            is_causal=False)

    def forward(self, values, padding):
        return self.head(self.encode(values, padding)).squeeze(-1)

## 4. Train the encoder inside a training split

Both input and target scaling are fitted on training patients only. Target scaling improves
optimization; reported predictions and MSE are converted back to OFF score units.
Early stopping uses its own patient holdout, including independently fitted preprocessing.
The selected epoch count is then used to refit a fresh encoder on all outer training patients.

In [4]:
def train_network(training, seed, epochs, validation=None):
    torch.manual_seed(seed)
    network = VisitEncoder(training.tensors[0].shape[-1], **ENCODER).to(DEVICE)
    optimizer = torch.optim.AdamW(network.parameters(), lr=LEARNING_RATE,
                                 weight_decay=WEIGHT_DECAY)
    loader = DataLoader(training, batch_size=BATCH_SIZE, shuffle=True)
    best_loss, best_epoch, best_state, stale = np.inf, epochs, None, 0
    history = []

    for epoch in range(1, epochs + 1):
        network.train()
        total_error, count = 0.0, 0
        for values, targets, rows in loader:
            values, targets, rows = (t.to(DEVICE) for t in (values, targets, rows))
            valid = rows >= 0
            optimizer.zero_grad()
            prediction = network(values, ~valid)
            loss = nn.functional.mse_loss(prediction[valid], targets[valid])
            loss.backward()
            nn.utils.clip_grad_norm_(network.parameters(), 1.0)
            optimizer.step()
            total_error += loss.item() * valid.sum().item()
            count += valid.sum().item()
        history.append({"epoch": epoch, "train_MSE": total_error / count})

        if validation is not None:
            network.eval()
            total_error, count = 0.0, 0
            with torch.no_grad():
                for values, targets, rows in DataLoader(validation, batch_size=BATCH_SIZE):
                    values, targets, rows = (t.to(DEVICE) for t in (values, targets, rows))
                    valid = rows >= 0
                    prediction = network(values, ~valid)
                    total_error += ((prediction[valid] - targets[valid]) ** 2).sum().item()
                    count += valid.sum().item()
            stop_loss = total_error / count
            history[-1]["stop_MSE"] = stop_loss
            if stop_loss < best_loss:
                best_loss, best_epoch, stale = stop_loss, epoch, 0
                best_state = deepcopy(network.state_dict())
            else:
                stale += 1
            if stale >= PATIENCE:
                break

    if best_state is not None:
        network.load_state_dict(best_state)
    return network.eval(), best_epoch, pd.DataFrame(history)


def fit_encoder(raw, targets, features, seed):
    fit_idx, stop_idx = next(GroupKFold(
        n_splits=5, shuffle=True, random_state=seed
    ).split(raw, groups=raw["patient_id"]))
    fit_raw, stop_raw = raw.iloc[fit_idx], raw.iloc[stop_idx]
    inner_prep = clone(encoder_prep)
    fit_features = inner_prep.fit_transform(fit_raw)
    stop_features = inner_prep.transform(stop_raw)
    scaler = StandardScaler().fit(fit_features)
    mean = targets.iloc[fit_idx].mean()
    scale = max(targets.iloc[fit_idx].std(ddof=0), 1e-8)
    training = patient_sequences(fit_raw, scaler.transform(fit_features),
                                 (targets.iloc[fit_idx] - mean) / scale)
    stopping = patient_sequences(stop_raw, scaler.transform(stop_features),
                                 (targets.iloc[stop_idx] - mean) / scale)
    _, epochs, history = train_network(training, seed, MAX_EPOCHS, stopping)
    history[["train_MSE", "stop_MSE"]] *= scale ** 2

    # Fresh model and scaling, now fitted on every patient in the outer training split.
    scaler = StandardScaler().fit(features)
    mean, scale = targets.mean(), max(targets.std(ddof=0), 1e-8)
    training = patient_sequences(raw, scaler.transform(features), (targets - mean) / scale)
    network, _, _ = train_network(training, seed, epochs)
    network.requires_grad_(False)
    return dict(network=network, scaler=scaler, target_mean=mean, target_scale=scale,
                epochs=epochs, history=history)

## 5. Extract frozen embeddings in the original row order

The prediction head stays available for diagnostics; HistGB receives only the contextual
embeddings, with no CLS token, pooling or predicted-score column.

In [5]:
@torch.no_grad()
def encoder_outputs(fitted, raw, features):
    features = fitted["scaler"].transform(features)
    sequences = patient_sequences(raw, features)
    network = fitted["network"].eval()
    embeddings = np.empty((len(raw), network.head.in_features), dtype=np.float32)
    predictions = np.empty(len(raw), dtype=np.float32)
    for values, _, rows in DataLoader(sequences, batch_size=BATCH_SIZE):
        padding = rows < 0
        hidden = network.encode(values.to(DEVICE), padding.to(DEVICE))
        prediction = network.head(hidden).squeeze(-1).cpu()
        valid_rows = rows[~padding].numpy()
        embeddings[valid_rows] = hidden.cpu()[~padding].numpy()
        predictions[valid_rows] = prediction[~padding].numpy()
    embeddings = pd.DataFrame(embeddings, index=raw.index,
                              columns=[f"enc_{i:02d}" for i in range(embeddings.shape[1])])
    predictions = predictions * fitted["target_scale"] + fitted["target_mean"]
    return embeddings, pd.Series(predictions, index=raw.index)

## 6. Fold-specific Encoder–HistGB validation

Use five disjoint GroupKFold patient folds with seeds 42, 43 and 44 (15 runs). Preprocessing is fitted
again in each split and shared by the unchanged baseline and augmented model. For each seed,
train a fresh encoder on that split's training patients, freeze it, append its embeddings and
fit a fresh clone of the exact same HistGB. No embeddings are combined across splits.
Negative delta_MSE means the embeddings help. The standard deviation describes variability across splits and seeds, not uncertainty about the true test performance.

In [6]:
cv = GroupKFold(n_splits=5, shuffle=True, random_state=42)
scores = []
with threadpool_limits(limits=3):
    for fold, (train_idx, val_idx) in enumerate(cv.split(X, y, groups=X["patient_id"]), 1):
        raw_train, raw_val = X.iloc[train_idx], X.iloc[val_idx]
        yt, yv = y.iloc[train_idx], y.iloc[val_idx]
        prep = clone(baseline[:-1])
        train = prep.fit_transform(raw_train, yt)
        validation = prep.transform(raw_val)
        encoder_transform = clone(encoder_prep[2:])
        encoder_train = encoder_transform.fit_transform(prep[:2].transform(raw_train))
        encoder_val = encoder_transform.transform(prep[:2].transform(raw_val))
        original = clone(baseline[-1]).fit(train, yt)
        baseline_mse = mean_squared_error(yv, original.predict(validation))

        for seed in SEEDS:
            fitted_encoder = fit_encoder(raw_train, yt, encoder_train, seed)
            z_train, _ = encoder_outputs(fitted_encoder, raw_train, encoder_train)
            z_val, encoder_prediction = encoder_outputs(fitted_encoder, raw_val, encoder_val)
            augmented = clone(baseline[-1]).fit(train.join(z_train), yt)
            mse = mean_squared_error(yv, augmented.predict(validation.join(z_val)))
            scores.append(dict(fold=fold, seed=seed, epochs=fitted_encoder["epochs"],
                               HistGB=baseline_mse, Transformer=mean_squared_error(yv, encoder_prediction),
                               Encoder_HistGB=mse, delta_MSE=mse - baseline_mse))
            print(f"Fold {fold}/{cv.n_splits}, seed {seed}, {fitted_encoder['epochs']} epochs: "
                  f"HistGB {baseline_mse:.4f} → Encoder–HistGB {mse:.4f}")
scores = pd.DataFrame(scores)
scores

Fold 1/5, seed 42, 31 epochs: HistGB 11.9597 → Encoder–HistGB 8.2217
Fold 1/5, seed 43, 64 epochs: HistGB 11.9597 → Encoder–HistGB 7.5551
Fold 1/5, seed 44, 63 epochs: HistGB 11.9597 → Encoder–HistGB 7.7476
Fold 2/5, seed 42, 38 epochs: HistGB 11.7222 → Encoder–HistGB 7.9804
Fold 2/5, seed 43, 24 epochs: HistGB 11.7222 → Encoder–HistGB 8.2914
Fold 2/5, seed 44, 35 epochs: HistGB 11.7222 → Encoder–HistGB 8.1054
Fold 3/5, seed 42, 44 epochs: HistGB 11.2770 → Encoder–HistGB 8.0161
Fold 3/5, seed 43, 35 epochs: HistGB 11.2770 → Encoder–HistGB 8.2571
Fold 3/5, seed 44, 33 epochs: HistGB 11.2770 → Encoder–HistGB 8.1398
Fold 4/5, seed 42, 28 epochs: HistGB 12.1577 → Encoder–HistGB 8.8475
Fold 4/5, seed 43, 34 epochs: HistGB 12.1577 → Encoder–HistGB 8.7121
Fold 4/5, seed 44, 37 epochs: HistGB 12.1577 → Encoder–HistGB 8.5774
Fold 5/5, seed 42, 40 epochs: HistGB 11.5505 → Encoder–HistGB 7.9771
Fold 5/5, seed 43, 23 epochs: HistGB 11.5505 → Encoder–HistGB 8.2879
Fold 5/5, seed 44, 56 epochs: Hist

,fold,seed,epochs,HistGB,Transformer,Encoder_HistGB,delta_MSE
0,1,42,31,11.959653,9.595669,8.221731,-3.737922
1,1,43,64,11.959653,7.624696,7.555070,-4.404583
2,1,44,63,11.959653,7.645369,7.747550,-4.212103
3,2,42,38,11.722212,8.414085,7.980404,-3.741808
4,2,43,24,11.722212,9.237500,8.291445,-3.430767
5,2,44,35,11.722212,9.898896,8.105388,-3.616824
6,3,42,44,11.277018,10.104226,8.016099,-3.260918
7,3,43,35,11.277018,9.215854,8.257103,-3.019914
8,3,44,33,11.277018,10.279204,8.139809,-3.137209
9,4,42,28,12.157750,10.646903,8.847537,-3.310213


In [7]:
summary = scores[["HistGB", "Transformer", "Encoder_HistGB"]].agg(
    ["mean", lambda s: s.std(ddof=0)]
).T
summary.columns = ["MSE", "std"]
summary["RMSE"] = np.sqrt(summary["MSE"])
summary["gain_MSE_pct"] = 100 * (1 - summary["MSE"] / summary.loc["HistGB", "MSE"])
summary

,MSE,std,RMSE,gain_MSE_pct
HistGB,11.733420,0.307616,3.425408,0.000000
Transformer,9.310657,0.917198,3.051337,20.648401
Encoder_HistGB,8.170199,0.339384,2.858356,30.368142


## Alternatives to test later

- Vary encoder depth, width and heads, or replace the encoder architecture.
- Predict ON/OFF measurements with this encoder or a separate encoder.
- Feed HistGB the encoder's final prediction instead of, or alongside, its embeddings.